# Backtesting With Leverage: Why It Lies
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nikshev/unstuck/blob/main/episodes/q00-backtester-leverage/q00_backtester_leverage.ipynb)

One 10x long on Bitcoin, opened on 3 February 2020. We backtest it on raw Binance data and check
the date and price at which a real exchange would have liquidated it.
Education, not financial advice.

## 1. Install
**freqtrade** is the backtester; **backtesting.py** is the one we compare it with at the end.

In [ ]:
!pip install -q freqtrade==2026.10 backtesting==0.6.6

## 2. Imports
Standard library for downloading and checking files, pandas for tables, matplotlib for charts; library notices hidden.

In [ ]:
import hashlib, io, json, re, zipfile, urllib.request
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import warnings
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore", category=UserWarning)

## 3. Download one file
Binance publishes every month of futures data at **data.binance.vision** — no account, no API key.

In [ ]:
BASE = "https://data.binance.vision/data/futures/um/monthly"
LIST = "https://s3-ap-northeast-1.amazonaws.com/data.binance.vision"

def fetch(url, tries=4):
    for attempt in range(tries):
        try:
            with urllib.request.urlopen(url, timeout=60) as r:
                return r.read()
        except OSError:
            if attempt == tries - 1:
                raise

## 4. Refuse a corrupt file
Next to every zip Binance puts a `.CHECKSUM`: its SHA-256. We compare it with the hash of what we received. Uses `fetch` from section 3.

In [ ]:
def verified_zip(url):
    blob = fetch(url)
    expected = fetch(url + ".CHECKSUM").decode().split()[0]
    actual = hashlib.sha256(blob).hexdigest()
    if actual != expected:
        raise ValueError(f"checksum mismatch: {url}")
    return zipfile.ZipFile(io.BytesIO(blob))

## 5. Read the CSV inside
Older files have no header row, newer ones do — keep only rows whose first column is a number.

In [ ]:
def read_csv(z):
    df = pd.read_csv(z.open(z.namelist()[0]), header=None)
    return df[pd.to_numeric(df[0], errors="coerce").notna()].astype(float)

## 6. Which months exist
`months` lists the months we ask for; `archived` lists the files Binance actually has — a delisted coin just stops early. Uses `fetch` and `LIST` from section 3.

In [ ]:
def months(start, end):
    return [p.strftime("%Y-%m") for p in pd.period_range(start, end, freq="M")]

def archived(prefix):
    xml = fetch(f"{LIST}?prefix=data/futures/um/monthly/{prefix}/&delimiter=/").decode()
    return set(re.findall(r"<Key>[^<]*/([^/<]+\.zip)</Key>", xml))

## 7. Many files at once
Sixteen downloads in parallel, results kept in month order. Uses `read_csv` and `verified_zip` from sections 4–5.

In [ ]:
def download_all(urls):
    with ThreadPoolExecutor(max_workers=16) as pool:
        return list(pool.map(lambda u: read_csv(verified_zip(u)), urls))

## 8. Candles
Daily candles for trading and hourly **mark-price** candles (what funding is charged on). Uses `archived`, `months` (6), `download_all` (7), `BASE` (3).

In [ ]:
def klines(symbol, interval, start, end, kind="klines"):
    have = archived(f"{kind}/{symbol}/{interval}")
    names = [f"{symbol}-{interval}-{m}.zip" for m in months(start, end)]
    parts = download_all([f"{BASE}/{kind}/{symbol}/{interval}/{n}" for n in names if n in have])
    df = pd.concat([p.iloc[:, :6] for p in parts], ignore_index=True)
    df.columns = ["date", "open", "high", "low", "close", "volume"]
    df["date"] = pd.to_datetime(df["date"], unit="ms", utc=True)
    return df.drop_duplicates("date").sort_values("date").reset_index(drop=True)

## 9. Funding rates
Every 8 hours longs pay shorts (or the reverse) a small percentage of the position. Uses `archived`, `months` (6), `download_all` (7), `BASE` (3).

In [ ]:
def funding(symbol, start, end):
    have = archived(f"fundingRate/{symbol}")
    names = [f"{symbol}-fundingRate-{m}.zip" for m in months(start, end)]
    parts = download_all([f"{BASE}/fundingRate/{symbol}/{n}" for n in names if n in have])
    df = pd.concat([p.iloc[:, [0, 2]] for p in parts], ignore_index=True)
    df.columns = ["date", "funding_rate"]
    df["date"] = pd.to_datetime(df["date"], unit="ms", utc=True).dt.round("h")
    return df.drop_duplicates("date").sort_values("date").reset_index(drop=True)

## 10. Save where freqtrade looks
Four files per coin: trading candles, hourly candles (so freqtrade can see *when* in the day a price was hit), mark price, funding. Uses `klines` and `funding` from sections 8–9.

In [ ]:
DATA = Path("user_data/data/binance/futures")

def save_for_freqtrade(symbol, timeframe, start, end):
    DATA.mkdir(parents=True, exist_ok=True)
    pair = symbol.replace("USDT", "_USDT_USDT")
    klines(symbol, timeframe, start, end).to_feather(DATA / f"{pair}-{timeframe}-futures.feather")
    klines(symbol, "1h", start, end).to_feather(DATA / f"{pair}-1h-futures.feather")
    klines(symbol, "1h", start, end, "markPriceKlines").to_feather(DATA / f"{pair}-1h-mark.feather")
    funding(symbol, start, end).to_feather(DATA / f"{pair}-1h-funding_rate.feather")

## 11. Bitcoin, January–December 2020
Every file checked against its SHA-256 on the way in. Uses `save_for_freqtrade` and `DATA` from section 10.

In [ ]:
save_for_freqtrade("BTCUSDT", "1d", "2020-01", "2020-12")
btc = pd.read_feather(DATA / "BTC_USDT_USDT-1d-futures.feather").set_index("date")
btc_1h = pd.read_feather(DATA / "BTC_USDT_USDT-1h-futures.feather").set_index("date")
print(len(btc), "days,", len(btc_1h), "hours,", btc.index[0].date(), "to", btc.index[-1].date())

## 12. The trade we test
Buy at the open on 3 February 2020 with 10x leverage, then hold. Uses `btc` from section 11.

In [ ]:
entry = btc.loc["2020-02-03", "open"]
window = btc.loc["2020-02-01":"2020-03-15"]
print("entry price:", entry)
print("lowest low :", window["low"].min(), "on", window["low"].idxmin().date())
plt.figure(figsize=(11, 5))
plt.plot(window.index, window["close"], label="BTC close")
plt.plot(window.index, window["low"], ":", label="BTC daily low")
plt.axhline(entry, color="gray", ls="--", label="entry")
plt.legend(); plt.title("BTCUSDT perpetual, Feb–Mar 2020"); plt.show()

## 13. Backtest settings
Futures, **isolated** margin (a position can lose only its own margin), no liquidation buffer, 1,000 USDT per trade, 0.05% fee, daily signals checked against hourly candles.

In [ ]:
config = {
    "trading_mode": "futures", "margin_mode": "isolated", "liquidation_buffer": 0.0,
    "stake_currency": "USDT", "stake_amount": 1000, "dry_run_wallet": 10000,
    "max_open_trades": 1, "timeframe": "1d", "timeframe_detail": "1h", "dry_run": True, "fee": 0.0005,
    "entry_pricing": {"price_side": "same", "use_order_book": True, "order_book_top": 1},
    "exit_pricing": {"price_side": "same", "use_order_book": True, "order_book_top": 1},
    "exchange": {"name": "binance", "pair_whitelist": ["BTC/USDT:USDT"]},
    "pairlists": [{"method": "StaticPairList"}],
}
json.dump(config, open("config.json", "w"), indent=1)

## 14. The strategy: one long, 10x
A signal on the candle of 2 February fills at the next open — 3 February. freqtrade insists on a stoploss; −99% is as far as it goes.

In [ ]:
%%writefile user_data/strategies/OneTrade.py
import pandas as pd
from freqtrade.strategy import IStrategy

class OneTrade(IStrategy):
    timeframe = "1d"
    can_short = True
    minimal_roi = {"0": 100}
    stoploss = -0.99
    SIDE, DAY = "long", "2020-02-02"

    def populate_indicators(self, df, metadata):
        return df

    def populate_entry_trend(self, df, metadata):
        df.loc[df["date"] == pd.Timestamp(self.DAY, tz="UTC"), "enter_" + self.SIDE] = 1
        return df

    def populate_exit_trend(self, df, metadata):
        return df

    def leverage(self, pair, **kwargs):
        return 10

## 15. Read the result
freqtrade writes every trade into a zip; `load_backtest_data` reads the newest one back as a table.

In [ ]:
from freqtrade.data.btanalysis import load_backtest_data

def last_trade():
    t = load_backtest_data(max(Path("user_data/backtest_results").glob("*.zip"),
                               key=lambda p: p.stat().st_mtime)).iloc[0]
    for k in ["open_date", "open_rate", "amount", "close_date", "close_rate", "exit_reason",
              "stake_amount", "funding_fees", "profit_abs"]:
        print(f"{k:>13}: {t[k]}")
    print(f"{'loss/margin':>13}: {-t['profit_abs'] / t['stake_amount']:.0%}")
    return t

## 16. Run it
`--cache none`: never reuse an older result. Uses `config.json` (section 13), `OneTrade` (14), `last_trade` (15).

In [ ]:
!freqtrade backtesting --cache none -c config.json -s OneTrade --timerange 20200101-20200501 --export trades > run.log 2>&1
default = last_trade()

## 17. Where a real exchange liquidates
Isolated long: $P_{liq} = E \cdot \dfrac{1 - 1/L}{1 - MMR}$ — E entry price, L leverage, MMR maintenance margin rate (Binance BTCUSDT, first tier: 0.4%). Uses `entry` (section 12), `btc_1h` (11).

In [ ]:
L, MMR = 10, 0.004
liq = entry * (1 - 1 / L) / (1 - MMR)
below = btc_1h.loc["2020-02-03":]
print("liquidation price:", round(liq, 2))
print("first hourly low below it:", below[below["low"] < liq].index[0])

## 18. Funding, paid every 8 hours
Paid = rate × position size × mark price, summed from the entry to the exit freqtrade reported in section 16 (`default`, which also holds the size it bought). Uses `DATA` (10), `entry` (12), `L` (17).

In [ ]:
rates = pd.read_feather(DATA / "BTC_USDT_USDT-1h-funding_rate.feather").set_index("date")["funding_rate"]
mark = pd.read_feather(DATA / "BTC_USDT_USDT-1h-mark.feather").set_index("date")["open"]
amount = default["amount"]
paid = (rates * amount * mark).dropna().loc[default["open_date"]:default["close_date"]]
print("payments:", len(paid), " total paid:", round(paid.sum(), 2), "USDT")
print("share of the margin:", f"{paid.sum() / (amount * entry / L):.0%}")

## 19. Funding eats the margin, so liquidation creeps up
Margin left $m = 1 - \text{funding paid}/\text{margin}$; then $P_{liq} = E \cdot \dfrac{1 - m/L}{1 - MMR}$. Recomputed after every payment and held until the next one. Uses `paid`, `amount` (18), `entry`, `window` (12), `L`, `MMR`, `liq`, `below` (17).

In [ ]:
margin = amount * entry / L
m = 1 - paid.cumsum() / margin
liq_now = (entry * (1 - m / L) / (1 - MMR)).reindex(below.index, method="ffill").dropna()
first = below.loc[liq_now.index][below["low"] < liq_now].index[0]
print("liquidated at", first, "price", round(liq_now[first], 2))
plt.figure(figsize=(11, 5))
plt.plot(window.index, window["low"], label="BTC daily low")
plt.plot(liq_now.index, liq_now, label="liquidation price, with funding")
plt.axhline(liq, color="gray", ls="--", label="liquidation price, no funding")
plt.legend(); plt.title("Funding pulls the liquidation price up"); plt.show()

## 20. Teach freqtrade to liquidate
`custom_stoploss` runs on every candle; `stoploss_from_absolute` turns our price into the relative stop freqtrade wants. Subclass of `OneTrade` (section 14).

In [ ]:
%%writefile user_data/strategies/Liquidated.py
from freqtrade.strategy import stoploss_from_absolute
from OneTrade import OneTrade

MMR = 0.004

class Liquidated(OneTrade):
    use_custom_stoploss = True

    def custom_stoploss(self, pair, trade, current_time, current_rate, current_profit, after_fill, **kwargs):
        m = 1 + (trade.funding_fees or 0) / trade.stake_amount
        if trade.is_short:
            liq = trade.open_rate * (1 + m / trade.leverage) / (1 + MMR)
        else:
            liq = trade.open_rate * (1 - m / trade.leverage) / (1 - MMR)
        return stoploss_from_absolute(liq, current_rate, is_short=trade.is_short, leverage=trade.leverage)

## 21. Run it again
Same data, same trade — now with the liquidation price. Uses `config.json` (13), `Liquidated` (20), `last_trade` (15).

In [ ]:
!freqtrade backtesting --cache none -c config.json -s Liquidated --timerange 20200101-20200501 --export trades > run.log 2>&1
fixed = last_trade()

## 22. The same mistake, short
A 10x short on 8 October 2020, a week before Bitcoin's run from 10,600 to 13,000. Isolated short: $P_{liq} = E \cdot \dfrac{1 + m/L}{1 + MMR}$. Subclass of `Liquidated` (section 20).

In [ ]:
%%writefile user_data/strategies/LiquidatedShort.py
from Liquidated import Liquidated

class LiquidatedShort(Liquidated):
    SIDE, DAY = "short", "2020-10-07"

## 23. Run the short
Uses `config.json` (13), `LiquidatedShort` (22), `last_trade` (15).

In [ ]:
!freqtrade backtesting --cache none -c config.json -s LiquidatedShort --timerange 20201001-20201231 --export trades > run.log 2>&1
short = last_trade()

## 24. The most popular Python backtester, same trade
backtesting.py with `margin=0.1` (10x) checks the margin on closing prices only. Uses `btc` (section 11).

In [ ]:
from backtesting import Strategy
from backtesting.lib import FractionalBacktest

class OneLong(Strategy):
    def init(self):
        pass
    def next(self):
        if self.data.index[-1] == pd.Timestamp("2020-02-02") and not self.position:
            self.buy(size=0.999)

prices = btc.loc[:"2020-05-01"].rename(columns=str.capitalize).tz_localize(None)
stats = FractionalBacktest(prices, OneLong, cash=1000, margin=0.1, commission=0.0005,
                           finalize_trades=True).run()
print(stats._trades[["EntryTime", "EntryPrice", "ExitTime", "ExitPrice", "PnL"]].to_string())

## 25. Four answers to one question
The same 10x long, the same data. Uses `stats` (24), `default` (16), `fixed` (21), `liq`, `below` (17).

In [ ]:
bt = stats._trades.iloc[0]
when = lambda x: pd.Timestamp(x).strftime("%Y-%m-%d %H:%M")
table = pd.DataFrame([
    ["backtesting.py", when(bt["ExitTime"]), bt["ExitPrice"], bt["PnL"]],
    ["freqtrade, default", when(default["close_date"]), default["close_rate"], default["profit_abs"]],
    ["price alone, no funding", when(below[below["low"] < liq].index[0]), liq, None],
    ["freqtrade + liquidation + funding", when(fixed["close_date"]), fixed["close_rate"], fixed["profit_abs"]],
], columns=["model", "closed", "price", "profit USDT"]).round(2)
print(table.to_string(index=False))